In [1]:


#------------------------------------------------ Begin_Librairie ----------------------------------------

import pandas as pd 

from time import sleep

import datetime

from pandas import ExcelWriter

import re

import requests

import pdfplumber

import os

from bs4 import BeautifulSoup

from selenium import webdriver

from selenium.webdriver.common.by import By

from selenium.webdriver.common.keys import Keys

from webdriver_manager.chrome import ChromeDriverManager

from selenium.webdriver.chrome.options import Options

from selenium.webdriver.common.alert import Alert

from googletrans import Translator



AttributeError: module 'httpcore' has no attribute 'SyncHTTPTransport'

In [2]:
import googletrans
print(googletrans.__version__)
print(googletrans.__file__)

4.0.0-rc.1
c:\Users\wuj1\AppData\Local\Programs\Python\Python312\Lib\site-packages\googletrans\__init__.py


In [3]:
# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'CN NFRA' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.0")

now=datetime.datetime.now()

filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"
#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

writer = ExcelWriter(filename, engine='openpyxl')

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process



if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)

Running CN NFRA Web Scraping Tool v.1.0


In [4]:
# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------



#Starting Chrome driver, set to download files in tempfolder
#Try to download the insecure file in 



chrome_options = Options()
chrome_options.add_argument("--window-size=1920,1080")
chrome_options.add_argument("--allow-running-insecure-content")  # Allow insecure content

chrome_options.add_experimental_option("prefs", {
    "download.default_directory": tempfolder,
    "download.prompt_for_download": False,
    "download.directory_upgrade": True,
    "safebrowsing.enabled": True
})

driver = webdriver.Chrome(options=chrome_options)
driver.maximize_window()

session=requests.Session()
session.verify = False

In [5]:
#------------------------------------------------ Begin_Fouction ----------------------------------------


def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict


# Initialize the translator
translator = Translator()
def translate_text(text):
    return translator.translate(text, src='zh-cn', dest='en').text


def check_dowload_files(tempfolder, fileType, wait_time=10):

    for time in range(wait_time):

        if len([ele for ele in os.listdir(tempfolder) if '.crdownload' not in ele and '.tmp' not in ele]) != 0 :

            print(f"[INFO] : - {fileType} file = {os.listdir(tempfolder)[0]}")

            break

        else:

            print(f"[INFO] : - Download {fileType} file ... (wait {time*2}/{wait_time*2} s)")

            sleep(2)

    else:

        raise Exception(f'[ERROR] : - Failed to Download {fileType} file. Run Script again' )

In [ ]:
# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict = { 'CN NFRA 1': 'https://www.nfra.gov.cn/cn/view/pages/zhengwuxinxi/zhengfuxinxi.html#1',
            'CN NFRA 2': 'https://www.nfra.gov.cn/cn/view/pages/zhengwuxinxi/zhengfuxinxi.html#1',
            'CN NFRA 3': 'https://www.nfra.gov.cn/cn/view/pages/zhengwuxinxi/zhengfuxinxi.html#1',
            'CN NFRA 4': 'https://www.nfra.gov.cn/cn/view/pages/zhengwuxinxi/zhengfuxinxi.html#1',
            'CN NFRA 5': 'https://www.nfra.gov.cn/cn/view/pages/zhengwuxinxi/zhengfuxinxi.html#1',

            }

Typology ={

            'CN NFRA 1': 'List of Banking Financial Institutions',
            'CN NFRA 2': 'List of Branches of Foreign Banks',
            'CN NFRA 3': 'List of Insurance Institutions',
            'CN NFRA 4': 'List of Foreign Reinsurance Company Branches',
            'CN NFRA 5': 'List of Financial Holding Companies',

}

KeyWords = {
            'CN NFRA 1': '银行业金融机构法人名单',
            'CN NFRA 2': '外国及港澳台银行分行名单',
            'CN NFRA 3': '保险机构法人名单',
            'CN NFRA 4': '外国再保险公司分公司名单',
            'CN NFRA 5': '金融控股公司法人名单',

}

sqldict = {'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
         'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
         'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
         'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
         'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
         'Phone - Mother company': [], 'Check': []}

processdate = now.strftime('%Y-%m-%d')

In [ ]:



for reg in regdict:

    print(f'Working with list {reg}')

    driver.get(regdict[reg])

    input_element = driver.find_element(By.ID, 'search')

    # Enter the search term



    input_element.send_keys(KeyWords[reg])

    print(f'Input Chinese Keywords {KeyWords[reg]}')

    # Submit the form

    sleep(3)

    input_element.send_keys(Keys.RETURN)

    sleep(3)

    window_handles = driver.window_handles

    if len(window_handles)>1:

        driver.switch_to.window(window_handles[-1])

    else:

        print('Can not open the second Tab')    

    sleep(3)

    soup = BeautifulSoup(driver.page_source, 'html.parser')

    sleep(3)

    search_result = soup.find('div',class_='jiansuo-right')

    sleep(3)

    document_url = search_result.find(

        'div', class_ = 'jiansuo-right-result'

    ).find('li',class_ = 'jiansuo-right-result-list').find('a')['href']



    driver.get('https://www.nfra.gov.cn'+document_url)  

    sleep(3)

    soup = BeautifulSoup(driver.page_source, 'html.parser')

    publish_date = soup.find('div',class_='pages-date').find('span' ).text.split('：')[-1] #CHINESE : English character is invalid

    attach_file = soup.find('div',class_ = 'wenzhang-fujian').find('a')['href']

    driver.get('https://www.nfra.gov.cn'+attach_file)

    tables = []

    sleep(5)

    dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]

    

    with pdfplumber.open(dl_files[0]) as pdf:

        for page in pdf.pages:

            table = page.extract_table()

            #print(table)

            tables.append(table)

    df = pd.DataFrame({})

    for index1, table in enumerate(tables):

        if index1 == 0 :

            #print(index1)    

            for index, infos in enumerate(table):

                if index ==0:

                    #print(infos)

                    df = pd.DataFrame(columns=infos)

                else:

                    df.loc[len(df)] = infos

        else:

            for index, infos in enumerate(table):

                df.loc[(len(df))]=infos     

    for ENname,CNname,internalId in zip(df['英文全称'],df['中文全称'],df['机构编码']):

    

        #print(CNname,ENname,internalId)
        #sqldict['Name_2'].append(CNname)
        sqldict['Name'].append(CNname)
        
        if '无' or '' in ENname:

            print(CNname,ENname)
            
            sleep(3)

            # Retry logic for translation
            max_retries = 3
            for attempt in range(max_retries):
                try:
                    trans_name = translate_text(CNname)
                    break
                except Exception as e:
                    print(f"Translation failed on attempt {attempt+1}: {e}")
                    sleep(2)
                    trans_name = CNname  # fallback to original name if all retries fail
            else:
                print(f"Translation failed after {max_retries} attempts, using original name.")
                trans_name = CNname

            #print(trans_name)

            sqldict['Name - Mother Company'].append(trans_name)

        else:

            ENname = ENname.replace('\n', ' ')

            sqldict['Name - Mother Company'].append(ENname)

        sqldict['ListProcessDate'].append(processdate)    

        sqldict['RegulationType'].append('Regulated')

        sqldict['RegCtry'].append(reg.split()[0])

        sqldict['RegCode'].append(reg.split()[1])

        sqldict['ListCode'].append(reg.split()[2])

        sqldict['ListName'].append(Typology[reg])

        sqldict['InternalID_1'].append(str(internalId))

        sqldict['InternalID_1_type'].append('Organization code')

        sqldict['RegulationDate'].append(publish_date)

    sqldict = bourange_same_length_array(sqldict)

    

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))


Working with list CN CSRC 1
Input Chinese Keywords 银行业金融机构法人名单
国家开发银行 China Development Bank
中国进出口银行 The Export-Import Bank of China
中国农业发展银行 Agricultural Development Bank of China
中国工商银行股份有限公司 INDUSTRIAL AND COMMERCIAL BANK OF CHINA
LIMITED
中国农业银行股份有限公司 AGRICULTURAL BANK OF CHINA LIMITED
中国银行股份有限公司 BANK OF CHINA LIMITED
中国建设银行股份有限公司 CHINA CONSTRUCTION BANK CORPORATION
交通银行股份有限公司 BANK OF COMMUNICATIONS CO., LTD.
中国邮政储蓄银行股份有限公司 POSTAL SAVINGS BANK OF CHINA Co., Ltd.
中信银行股份有限公司 CHINA CITIC BANK CORPORATION LIMITED
中国光大银行股份有限公司 China Everbright Bank Co., Ltd.
招商银行股份有限公司 CHINA MERCHANTS BANK CO., LTD.
上海浦东发展银行股份有限公司 Shanghai PuDong Development Bank CO.,
LTD.
中国民生银行股份有限公司 CHINA MINSHENG BANKING CORPORATION
LIMITED
华夏银行股份有限公司 HUA XIA BANK CO.,Limited
平安银行股份有限公司 Ping An Bank Co.,Ltd.
兴业银行股份有限公司 Industrial Bank Co., Ltd.
广发银行股份有限公司 China Guangfa Bank Co.,Ltd.
渤海银行股份有限公司 CHINA BOHAI BANK CO.,LTD.
浙商银行股份有限公司 CHINA ZHESHANG BANK CO.,LTD.
恒丰银行股份有限公司 EVERGROWING BANK CO., Limited
北京银行股份有限公司 Bank of

In [ ]:
# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------


os.chdir(scriptfolder)

df=pd.DataFrame(sqldict)

df  = df.drop_duplicates()

df.to_excel(writer, 'SQL Ready', index=False)

writer.save()

writer.close()

driver.quit()

sleep(3)
     

C:\Users\wuj1\AppData\Local\Temp\8\ipykernel_17856\2773079153.py:12: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(writer, 'SQL Ready', index=False)


AttributeError: 'OpenpyxlWriter' object has no attribute 'save'

In [ ]:
df['InternalID_1'].to_csv('id.csv')

In [ ]:
df.to_csv('list3_ver5.txt')